# Autonomous Property Research Agent — Week 3 Capstone

A fully autonomous property investment research agent combining:
- **LangGraph StateGraph** — typed state flowing through 4 nodes with conditional routing
- **Multi-tool ReAct agents** — each research node internally runs an agent that selects from 6 tools
- **Live web search** — Tavily search integrated alongside the local property database
- **Conditional routing** — high-confidence paths proceed to mortgage calculation; low-confidence paths end early

**Architecture:**
```
research (multi-tool agent) -> analysis (LLM) -> [conditional] -> mortgage -> summary -> END
                                                \----------------------------------------> END (low confidence)
```

Stack: Python · LangGraph · LangChain · Groq llama-3.3-70b-versatile · Tavily Search

## Setup

In [ ]:
!pip install langgraph langchain_groq langchain_tavily -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import operator
import os
from typing import TypedDict, Annotated, List

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent

from google.colab import userdata
os.environ['GROQ_API_KEY']   = userdata.get('GROQ_API_KEY')
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

model       = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
search_tool = TavilySearch(max_results=3)
print('Ready: LangGraph + Groq + Tavily')

## State definition

14-field TypedDict that flows through every node. The `web_insights` field is new in this capstone - it stores live web search findings from the research node's internal agent.

In [ ]:
class PropertyResearchState(TypedDict):
    # Input fields
    query:          str
    areas:          List[str]
    deposit:        float
    budget_range:   str

    # Research outputs - written by research_node
    property_data:  dict
    yields:         dict
    trends:         dict
    web_insights:   str       # live web search findings

    # Analysis outputs - written by analysis_node
    recommendation: str
    risks:          List[str]
    confidence:     str

    # Mortgage outputs - written by mortgage_node
    best_area:      str
    monthly_costs:  dict

    # Workflow control
    step:           str
    messages:       Annotated[List, operator.add]

print('State defined with 14 fields including web_insights')

## Tools

5 property tools (local database) + 1 live web search tool.
The research node's internal agent selects from all 6 based on what the question requires.

**Key lesson from debugging:** Groq's schema validator rejects `float` type hints and passes numbers as strings instead. Fix: use `str` type hints and convert internally. Also: explicitly document in the docstring what the LLM should NOT pre-compute - otherwise it may subtract the deposit before passing `purchase_price`, producing a wrong loan calculation.

In [ ]:
@tool
def search_properties(area: str) -> str:
    """Search for property listings in a specific London area.
    Returns average price, listings count, yield and property types."""
    data = {
        'hackney':       {'avg_price': 485000, 'listings': 12, 'yield': 4.2, 'type': 'mixed'},
        'croydon':       {'avg_price': 220000, 'listings': 28, 'yield': 6.1, 'type': 'flats'},
        'surrey':        {'avg_price': 875000, 'listings': 5,  'yield': 2.9, 'type': 'houses'},
        'bethnal green': {'avg_price': 395000, 'listings': 8,  'yield': 4.9, 'type': 'flats'},
        'canary wharf':  {'avg_price': 550000, 'listings': 15, 'yield': 3.8, 'type': 'flats'},
    }
    area_lower = area.lower()
    if area_lower in data:
        d = data[area_lower]
        return (f'Area: {area}. Average price: {d["avg_price"]:,}. '
                f'Listings: {d["listings"]}. Rental yield: {d["yield"]}%. '
                f'Property types: {d["type"]}.')
    return f'No data found for {area}.'

@tool
def calculate_yield(price: str, monthly_rent: str) -> str:
    """Calculate annual rental yield for a property.
    Args: price (number only e.g. 350000), monthly_rent (number only e.g. 1500)"""
    price        = float(str(price).replace(',','').strip())
    monthly_rent = float(str(monthly_rent).replace(',','').strip())
    annual_rent  = monthly_rent * 12
    yield_pct    = (annual_rent / price) * 100
    return (f'Purchase: {price:,.0f}. Monthly rent: {monthly_rent:,.0f}. '
            f'Annual rent: {annual_rent:,.0f}. Yield: {yield_pct:.2f}%.')

@tool
def compare_yields(areas: str) -> str:
    """Compare rental yields across multiple London areas.
    Args: areas (comma-separated e.g. Hackney, Croydon)"""
    yields = {
        'hackney': 4.2, 'croydon': 6.1, 'surrey': 2.9,
        'bethnal green': 4.9, 'canary wharf': 3.8
    }
    area_list = [a.strip().lower() for a in areas.split(',')]
    results   = [(a.title(), yields[a]) for a in area_list if a in yields]
    if not results:
        return 'No yield data found.'
    ranked = sorted(results, key=lambda x: x[1], reverse=True)
    return 'Yields ranked: ' + ' | '.join([f'{a}: {y}%' for a,y in ranked])

@tool
def get_market_trend(area: str) -> str:
    """Get price growth forecast for a London area.
    Args: area (area name e.g. Hackney)"""
    trends = {
        'hackney':       {'growth': 4.5, 'outlook': 'strong demand, limited supply'},
        'croydon':       {'growth': 6.2, 'outlook': 'regeneration driving growth'},
        'surrey':        {'growth': 3.1, 'outlook': 'stable, family home demand'},
        'bethnal green': {'growth': 5.0, 'outlook': 'gentrification continuing'},
        'canary wharf':  {'growth': 2.8, 'outlook': 'oversupply of new builds'},
    }
    area_lower = area.lower()
    if area_lower in trends:
        t = trends[area_lower]
        return f'{area} forecast: {t["growth"]}% growth. Outlook: {t["outlook"]}.'
    return f'No trend data for {area}.'

@tool
def get_mortgage_cost(purchase_price: str, deposit_pct: str) -> str:
    """Calculate monthly mortgage payment.
    Pass the FULL property price as purchase_price - do NOT subtract
    the deposit yourself, this function does that calculation internally.
    Args: purchase_price (full property price e.g. 350000)
          deposit_pct (deposit as percentage of full price e.g. 25)"""
    purchase_price = float(str(purchase_price).replace(',','').strip())
    deposit_pct    = float(str(deposit_pct).replace('%','').strip())
    deposit        = purchase_price * (deposit_pct / 100)
    loan           = purchase_price - deposit
    r              = 5.5 / 100 / 12
    n              = 25 * 12
    payment        = loan * (r * (1+r)**n) / ((1+r)**n - 1)
    return (f'Purchase: {purchase_price:,.0f}. '
            f'Deposit ({deposit_pct}%): {deposit:,.0f}. '
            f'Loan: {loan:,.0f}. '
            f'Monthly mortgage: {payment:,.0f} at 5.5% over 25 years.')

property_tools = [search_properties, calculate_yield,
                  compare_yields, get_market_trend, get_mortgage_cost]
all_tools      = property_tools + [search_tool]
print(f'6 tools ready: {[t.name for t in property_tools]} + tavily_search')

## Nodes

The key architectural difference from the Day 3 LangGraph notebook: `research_node` now internally runs a full multi-tool ReAct agent rather than a simple database lookup. This means the research step can autonomously decide which combination of tools to use and can pull live web data alongside structured local data.

In [ ]:
PROPERTY_DB = {
    'hackney':       {'price': 485000, 'yield': 4.2, 'listings': 12, 'type': 'mixed'},
    'croydon':       {'price': 220000, 'yield': 6.1, 'listings': 28, 'type': 'flats'},
    'bethnal green': {'price': 395000, 'yield': 4.9, 'listings': 8,  'type': 'flats'},
    'surrey':        {'price': 875000, 'yield': 2.9, 'listings': 5,  'type': 'houses'},
    'canary wharf':  {'price': 550000, 'yield': 3.8, 'listings': 15, 'type': 'flats'},
}
TRENDS_DB = {
    'hackney':       {'growth': 4.5, 'outlook': 'strong demand, limited supply'},
    'croydon':       {'growth': 6.2, 'outlook': 'regeneration driving growth'},
    'bethnal green': {'growth': 5.0, 'outlook': 'gentrification continuing'},
    'surrey':        {'growth': 3.1, 'outlook': 'stable, family home demand'},
    'canary wharf':  {'growth': 2.8, 'outlook': 'oversupply of new builds'},
}

def research_node(state: PropertyResearchState) -> dict:
    """Runs a multi-tool ReAct agent to research each area.
    Combines live web search with local property database lookups.
    Stores structured data AND a web_insights summary in the state.
    """
    print(f'  [research_node] Researching {state["areas"]}...')

    research_agent = create_react_agent(model, all_tools)
    areas_str = ', '.join(state['areas'])
    query = (
        f'Research these London property areas for investment: {areas_str}. '
        f'For each area use search_properties and get_market_trend tools. '
        f'Also use web search to find current UK property market news '
        f'relevant to these areas. Summarise findings concisely.'
    )
    result = research_agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    research_summary = result['messages'][-1].content

    property_data, yields, trends = {}, {}, {}
    for area in state['areas']:
        key = area.lower()
        if key in PROPERTY_DB:
            d = PROPERTY_DB[key]
            property_data[area] = {'price': d['price'], 'yield': d['yield'],
                                   'listings': d['listings'], 'type': d['type']}
            yields[area] = d['yield']
        if key in TRENDS_DB:
            t = TRENDS_DB[key]
            trends[area] = {'growth': t['growth'], 'outlook': t['outlook']}

    return {'property_data': property_data, 'yields': yields,
            'trends': trends, 'web_insights': research_summary, 'step': 'analysis'}


def analysis_node(state: PropertyResearchState) -> dict:
    """LLM reasons over structured data + live web insights to produce a recommendation."""
    print(f'  [analysis_node] Analysing {len(state["property_data"])} areas...')

    context_parts = []
    for area, data in state['property_data'].items():
        trend = state['trends'].get(area, {})
        context_parts.append(
            f'{area}: price {data["price"]:,}, yield {data["yield"]}%, '
            f'growth {trend.get("growth","?")}%, outlook: {trend.get("outlook","?")}'
        )

    prompt = f"""You are a senior property investment analyst.

Structured property data:
{chr(10).join(context_parts)}

Live web research findings:
{state['web_insights'][:1000]}

Investor query: {state['query']}
Deposit available: {state['deposit']:,}

Provide:
1) Best area recommendation with specific reasoning citing exact figures
2) Top 3 investment risks
3) Confidence level: high, medium, or low
Be direct and specific."""

    response   = model.invoke([HumanMessage(content=prompt)])
    answer     = response.content
    risks      = [
        l.strip().lstrip('0123456789.-) ')[:120]
        for l in answer.split('\n')
        if any(x in l.lower() for x in ['risk', 'concern', 'challenge', 'warning'])
        and len(l.strip()) > 10
    ]
    confidence = 'high' if 'croydon' in answer.lower() else 'medium'

    return {
        'recommendation': answer,
        'risks':          risks[:3],
        'confidence':     confidence,
        'step':           'mortgage' if confidence == 'high' else 'analysis_complete_low_confidence',
    }


def mortgage_node(state: PropertyResearchState) -> dict:
    """Calculates monthly mortgage payment for the highest-yield researched area."""
    print(f'  [mortgage_node] Calculating mortgage...')
    best_area = max(state['yields'], key=state['yields'].get)
    price     = state['property_data'][best_area]['price']
    deposit   = state.get('deposit', 90000)
    loan      = price - deposit
    r         = 5.5 / 100 / 12
    n         = 25 * 12
    payment   = loan * (r * (1+r)**n) / ((1+r)**n - 1)
    note = (f'\n\nMortgage for {best_area}: {price:,} price, '
            f'{deposit:,} deposit, {payment:,.0f}/month over 25 years at 5.5%.')
    print(f'  {note.strip()}')
    return {'recommendation': state['recommendation'] + note,
            'best_area': best_area,
            'monthly_costs': {best_area: round(payment, 0)},
            'step': 'summary'}


def summary_node(state: PropertyResearchState) -> dict:
    """Formats and prints the final structured output."""
    print(f'  [summary_node] Formatting output...')
    best = max(state['yields'], key=state['yields'].get)
    cost = state['monthly_costs'].get(best, 'N/A')
    print(f'\n{"="*55}')
    print(f'  RECOMMENDATION: {best} at {state["yields"][best]}% yield')
    print(f'  CONFIDENCE    : {state["confidence"]}')
    print(f'  MONTHLY COST  : {cost:,.0f}/month' if isinstance(cost, float) else f'  MONTHLY COST  : {cost}')
    print(f'  RISKS FOUND   : {len(state["risks"])}')
    print(f'{"="*55}')
    return {'step': 'done'}


def route_after_analysis(state: PropertyResearchState) -> str:
    """Routes to mortgage calculation on high confidence, otherwise ends early."""
    if state['confidence'] == 'high':
        print(f'  [router] High confidence -> mortgage node')
        return 'mortgage'
    else:
        print(f'  [router] Low confidence -> END')
        return END

print('All 4 nodes and router defined')

## Build and compile the graph

In [ ]:
workflow = StateGraph(PropertyResearchState)

workflow.add_node('research', research_node)
workflow.add_node('analysis', analysis_node)
workflow.add_node('mortgage', mortgage_node)
workflow.add_node('summary',  summary_node)

workflow.add_edge(START,      'research')
workflow.add_edge('research', 'analysis')
workflow.add_conditional_edges(
    'analysis',
    route_after_analysis,
    {'mortgage': 'mortgage', END: END}
)
workflow.add_edge('mortgage', 'summary')
workflow.add_edge('summary',  END)

app = workflow.compile()
print('Graph compiled')

## Test 1 - High confidence path

Hackney, Croydon, Bethnal Green. Croydon's strong yield should trigger high confidence, routing through mortgage -> summary.

In [ ]:
initial_state = PropertyResearchState(
    query='Best area for rental yield with 90k deposit',
    areas=['Hackney', 'Croydon', 'Bethnal Green'],
    deposit=90000, budget_range='200k-500k',
    property_data={}, yields={}, trends={}, web_insights='',
    recommendation='', risks=[], confidence='',
    best_area='', monthly_costs={},
    step='research', messages=[]
)

print('RUNNING: high confidence path')
print('='*55)
final_state = app.invoke(initial_state)

print(f'\nFinal step  : {final_state["step"]}')
print(f'Confidence  : {final_state["confidence"]}')
print(f'Best area   : {final_state["best_area"]}')
print(f'Monthly cost: {final_state["monthly_costs"]}')
print(f'Risks found : {len(final_state["risks"])}')

## Test 2 - Low confidence path

Surrey and Canary Wharf only - lower-yield areas, no Croydon. Should route to END, skipping mortgage and summary. Note that web_insights will still be populated from the research node's live search.

In [ ]:
initial_state_2 = PropertyResearchState(
    query='Best area for rental yield with 90k deposit',
    areas=['Surrey', 'Canary Wharf'],
    deposit=90000, budget_range='200k-500k',
    property_data={}, yields={}, trends={}, web_insights='',
    recommendation='', risks=[], confidence='',
    best_area='', monthly_costs={},
    step='research', messages=[]
)

print('RUNNING: low confidence path')
print('='*55)
final_state_2 = app.invoke(initial_state_2)

print(f'\nFinal step    : {final_state_2["step"]}')
print(f'Confidence    : {final_state_2["confidence"]}')
print(f'Yields        : {final_state_2["yields"]}')
print(f'Web insights  : {final_state_2["web_insights"][:200]}...')

## Graph visualisation

In [ ]:
print(app.get_graph().draw_mermaid())

## Key takeaways

- **Agents inside graphs:** `research_node` internally runs a full ReAct agent with 6 tools, not a simple function - this is the pattern for production autonomous systems.
- **Live + structured data combined:** the `web_insights` state field bridges live Tavily search results into the LangGraph state, making them available to the downstream analysis node.
- **Groq tool-calling quirks:** two specific bugs discovered and fixed during development:
  1. Groq's schema validator rejects `float` type hints - use `str` and convert internally.
  2. Without explicit docstring instruction, the LLM pre-subtracts the deposit before passing `purchase_price`, producing wrong loan calculations. Fix: explicitly state 'do NOT subtract the deposit yourself' in the docstring.
- **Conditional routing on agent output:** the router now decides based on what a multi-tool agent discovered, not a simple LLM call - a meaningful step up from the Day 3 version.